# 03 — Sampling and Decoding: Controlling How LLMs Generate Text

> **Orbit -1 (Foundations)** · **Domain**: Foundations · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/castalia/blob/main/foundations/03_sampling_and_decoding.ipynb)

**Prerequisites**:
- [00_how_llms_work.ipynb](00_how_llms_work.ipynb) — The LLM pipeline and mental models

**What you will learn**:
- The full journey from logits to generated text — implemented from scratch
- Temperature scaling: the math and intuition behind controlling randomness
- Top-k sampling: truncating the distribution
- Top-p (nucleus) sampling: adaptive truncation
- Greedy decoding vs beam search: when determinism helps and when it hurts
- How to choose sampling parameters for different tasks

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Loading the model**
- Understand **Section 1 — From Logits to Probabilities: The Softmax Step**
- Understand **Section 2 — Temperature: Sharpening and Flattening the Distribution**
- Understand **Section 3 — Greedy Decoding: Always Picking the Top Token**
- Understand **Section 4 — Top-k Sampling: Truncating the Tail**


In [ ]:
# --- Google Colab Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch numpy matplotlib

import math
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "font.size": 9,
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Loading the model

We load **Qwen3-8B** in 4-bit quantisation so the model fits comfortably in a T4's
~15 GB of VRAM. All sampling and decoding code in this notebook is written
from scratch; we use the model only for forward passes to obtain logits.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Model dtype: {next(model.parameters()).dtype}")

---
## Section 1 — From Logits to Probabilities: The Softmax Step

When a language model processes a prompt, its final output is a vector of **logits** —
one raw, unnormalised score for every token in the vocabulary. These logits are
*not* probabilities: they can be negative, greater than one, and they do not sum
to one.

The **softmax** function converts logits into a valid probability distribution:

$$
P(\text{token}_i) = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}
$$

where $z_i$ is the logit for token $i$ and $V$ is the vocabulary size.

Let's see this in action.

In [ ]:
# ---------- Forward pass: extract raw logits ----------
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)

# logits shape: (batch_size, seq_len, vocab_size)
logits = outputs.logits[0, -1, :]  # last token position
print(f"Logits shape: {logits.shape}  (one score per vocabulary token)")
print(f"Min logit:  {logits.min().item():.2f}")
print(f"Max logit:  {logits.max().item():.2f}")
print(f"Sum:        {logits.sum().item():.2f}  ← clearly NOT a probability distribution")

# Top-10 logits
top10 = torch.topk(logits, 10)
print("\nTop-10 tokens by raw logit score:")
for score, idx in zip(top10.values, top10.indices):
    token = tokenizer.decode(idx.item())
    print(f"  {token!r:>12s}  logit = {score.item():+.2f}")

### Implementing softmax from scratch

The naïve formula $e^{z_i} / \sum e^{z_j}$ overflows for large logits. The
standard trick is to subtract $\max(z)$ before exponentiating — the result is
mathematically identical but numerically stable:

$$
P(\text{token}_i) = \frac{e^{z_i - z_{\max}}}{\sum_{j} e^{z_j - z_{\max}}}
$$

In [ ]:
def softmax_from_scratch(logits: torch.Tensor) -> torch.Tensor:
    """Numerically-stable softmax, implemented from first principles."""
    shifted = logits - logits.max()
    exps = torch.exp(shifted)
    return exps / exps.sum()

probs = softmax_from_scratch(logits)
print(f"Probabilities sum to: {probs.sum().item():.6f}")  # should be ~1.0
print(f"Min prob: {probs.min().item():.2e}")
print(f"Max prob: {probs.max().item():.4f}")

# Verify against PyTorch's implementation
probs_ref = torch.softmax(logits, dim=-1)
max_diff = (probs - probs_ref).abs().max().item()
print(f"Max difference vs torch.softmax: {max_diff:.2e}")

In [ ]:
# ---------- Visualise the top-30 token probabilities ----------
top30 = torch.topk(probs, 30)
tokens = [tokenizer.decode(idx.item()).strip() for idx in top30.indices]
values = top30.values.cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(tokens)), values, color="#4C72B0")
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=55, ha="right", fontsize=7)
ax.set_ylabel("Probability")
ax.set_title("Top-30 token probabilities after softmax — \"The capital of France is\"")
ax.set_ylim(0, values[0] * 1.15)
plt.tight_layout()
plt.show()

---
## Section 2 — Temperature: Sharpening and Flattening the Distribution

Temperature is the simplest and most important sampling knob. It divides the
logits *before* softmax:

$$
P(\text{token}_i) = \frac{e^{z_i / T}}{\sum_{j} e^{z_j / T}}
$$

| Temperature | Effect |
|---|---|
| $T \to 0^+$ | Distribution collapses to a spike on the top token (deterministic). |
| $T = 1.0$ | The model's "native" distribution — as trained. |
| $T > 1.0$ | Distribution flattens — unlikely tokens get more probability mass. |

**Intuition**: low temperature makes the model more *certain* (always picks the
obvious answer). High temperature makes the model *explore* (considers unlikely
tokens).

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by temperature and return probabilities."""
    if temperature <= 0:
        raise ValueError("Temperature must be > 0")
    scaled = logits / temperature
    return softmax_from_scratch(scaled)

# ---------- Side-by-side comparison ----------
temperatures = [0.1, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, T in zip(axes, temperatures):
    p = apply_temperature(logits, T)
    top = torch.topk(p, 20)
    toks = [tokenizer.decode(i.item()).strip() for i in top.indices]
    vals = top.values.cpu().numpy()
    ax.bar(range(len(toks)), vals, color="#4C72B0" if T == 1.0 else
           ("#DD5555" if T < 1 else "#55AA55"))
    ax.set_xticks(range(len(toks)))
    ax.set_xticklabels(toks, rotation=55, ha="right", fontsize=6)
    ax.set_title(f"T = {T}")
    ax.set_ylabel("Probability")

fig.suptitle("Temperature scaling — same logits, different distributions", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Temperature in practice: observing output diversity

Let's generate the same prompt 10 times at three different temperatures and
observe how repetitive or diverse the outputs become.

In [ ]:
def generate_with_temperature(prompt: str, temperature: float, max_new_tokens: int = 40,
                               num_samples: int = 10) -> List[str]:
    """Generate multiple completions using only temperature sampling (from scratch)."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    results = []

    for _ in range(num_samples):
        generated = input_ids.clone()
        for _ in range(max_new_tokens):
            with torch.no_grad():
                out = model(generated)
            step_logits = out.logits[0, -1, :]
            probs = apply_temperature(step_logits, temperature)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
            if next_token.item() == tokenizer.eos_token_id:
                break
        text = tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)
        results.append(text.strip())
    return results

prompt = "The best thing about learning to code is"
for T in [0.1, 0.7, 1.5]:
    print(f"\n{'='*70}")
    print(f"  Temperature = {T}")
    print(f"{'='*70}")
    completions = generate_with_temperature(prompt, T, max_new_tokens=30, num_samples=5)
    for i, c in enumerate(completions, 1):
        print(f"  [{i}] {c}")

### Temperature recommendations by task

| Task | Recommended T | Rationale |
|---|---|---|
| Factual QA / extraction | 0.0 – 0.3 | Minimise hallucination; pick the most likely answer. |
| Code generation | 0.2 – 0.4 | Code must be correct; slight variation for style. |
| Conversational chat | 0.5 – 0.8 | Natural-sounding variety without incoherence. |
| Creative writing | 0.8 – 1.2 | Broader vocabulary, more surprising word choices. |
| Brainstorming / ideation | 1.2 – 1.5 | Deliberately push the model beyond safe defaults. |

> **Rule of thumb**: start at T = 0.7 and adjust based on the failure mode you
> see. If outputs are repetitive or boring, raise T. If they are incoherent or
> hallucinating, lower T.

---
## Section 3 — Greedy Decoding: Always Picking the Top Token

The simplest decoding strategy: at every step, pick the token with the highest
probability (equivalently, the highest logit). No randomness at all.

```
for each step:
    logits = model(context)
    next_token = argmax(logits)
    context = context + [next_token]
```

**When it works**: factual answers, structured output (JSON, SQL), and any task
where there is essentially one correct answer.

**When it fails**: repetitive loops, degenerate text, lack of diversity. The
model can get stuck repeating the same phrase because each repetition is locally
probable.

In [ ]:
def greedy_decode(prompt: str, max_new_tokens: int = 80) -> str:
    """Greedy decoding implemented from scratch."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(generated)
        step_logits = out.logits[0, -1, :]
        next_token = step_logits.argmax().unsqueeze(0)       # ← argmax = greedy
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

# Good case: factual answer
print("=== Greedy on factual prompt ===")
print(greedy_decode("The chemical formula for water is"))

In [ ]:
# Greedy decoding failure mode: repetitive text
print("=== Greedy on open-ended prompt (watch for repetition) ===")
output = greedy_decode("Once upon a time in a land far away, there was a", max_new_tokens=150)
print(output)

---
## Section 4 — Top-k Sampling: Truncating the Tail

Top-k sampling keeps only the **k** most likely tokens and sets the probability
of all others to zero, then renormalises:

1. Sort tokens by probability (descending).
2. Keep the top-k tokens.
3. Set the rest to $-\infty$ (so they get probability 0 after softmax).
4. Renormalise and sample.

This prevents the model from sampling extremely unlikely tokens that would
produce gibberish. The trade-off: **k is a fixed number** regardless of how
confident the model is.

In [ ]:
def top_k_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Zero out all logits except the top-k. Returns filtered logits."""
    top_k_vals, _ = torch.topk(logits, k)
    threshold = top_k_vals[-1]               # smallest value that survives
    filtered = logits.clone()
    filtered[logits < threshold] = float("-inf")
    return filtered

# Show which tokens survive at different k values
print("Prompt: \"The capital of France is\"\n")
for k in [5, 50, 200]:
    filtered = top_k_filter(logits, k)
    p = softmax_from_scratch(filtered)
    survivors = (p > 0).sum().item()
    top5 = torch.topk(p, min(5, survivors))
    toks = [tokenizer.decode(i.item()).strip() for i in top5.indices]
    print(f"  k={k:>3d} → {survivors:>3d} surviving tokens.  Top-5: {toks}")

In [ ]:
# ---------- Visualise top-k filtering ----------
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, k in zip(axes, [5, 50, 200]):
    filtered = top_k_filter(logits, k)
    p = softmax_from_scratch(filtered)
    top_n = min(30, k)
    top = torch.topk(p, top_n)
    toks = [tokenizer.decode(i.item()).strip() for i in top.indices]
    vals = top.values.cpu().numpy()

    ax.bar(range(len(toks)), vals, color="#E8833A")
    ax.set_xticks(range(len(toks)))
    ax.set_xticklabels(toks, rotation=55, ha="right", fontsize=6)
    ax.set_title(f"Top-k = {k}  ({(p > 0).sum().item()} tokens)")
    ax.set_ylabel("Probability")

fig.suptitle("Top-k sampling — only the k most likely tokens survive", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### The problem with a fixed k

When the model is very confident (e.g., completing "The capital of France is"),
even k = 50 includes many irrelevant tokens. When the model is uncertain (e.g.,
creative writing), k = 50 may cut off perfectly reasonable alternatives.

**We need a way to adapt the candidate set to the model's confidence.** That's
what top-p sampling does.

---
## Section 5 — Top-p (Nucleus) Sampling: Adaptive Truncation

Instead of fixing the *number* of tokens, nucleus sampling fixes the
*cumulative probability mass*:

1. Sort tokens by probability (descending).
2. Compute the cumulative sum.
3. Include tokens until the cumulative probability first exceeds **p**.
4. Zero out the rest, renormalise, and sample.

The **nucleus** — the set of tokens that survive — automatically grows when the
model is uncertain and shrinks when the model is confident. This is why top-p
is the default in most production APIs.

In [ ]:
def top_p_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    """Nucleus (top-p) filtering implemented from scratch."""
    probs = softmax_from_scratch(logits)

    # Sort descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Find the cutoff: first position where cumulative prob exceeds p
    # We shift right by 1 so the token that *reaches* p is included
    sorted_mask = cumulative_probs - sorted_probs >= p   # True = remove
    sorted_probs[sorted_mask] = 0.0

    # Scatter back to original positions
    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(0, sorted_indices, sorted_probs)

    # Convert back to logits (for composability with temperature)
    filtered_logits = logits.clone()
    filtered_logits[filtered_probs == 0.0] = float("-inf")
    return filtered_logits

# Show adaptive nucleus size
print("Prompt: \"The capital of France is\"\n")
for p_val in [0.5, 0.9, 0.95, 0.99]:
    filtered = top_p_filter(logits, p_val)
    probs = softmax_from_scratch(filtered)
    nucleus_size = (probs > 0).sum().item()
    top3 = torch.topk(probs, min(3, nucleus_size))
    toks = [tokenizer.decode(i.item()).strip() for i in top3.indices]
    print(f"  p={p_val:.2f} → nucleus size: {nucleus_size:>4d}  Top-3: {toks}")

### Comparing top-k and top-p: nucleus size varies per step

The key advantage of top-p over top-k is that the nucleus **adapts to context**.
Let's generate a sequence and record the nucleus size at each decoding step.

In [ ]:
def track_nucleus_size(prompt: str, p: float = 0.9, max_steps: int = 50) -> Tuple[List[str], List[int]]:
    """Generate tokens with top-p and record the nucleus size at each step."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    generated = input_ids.clone()
    tokens_out, nucleus_sizes = [], []

    for _ in range(max_steps):
        with torch.no_grad():
            out = model(generated)
        step_logits = out.logits[0, -1, :]

        # Measure nucleus size
        probs = softmax_from_scratch(step_logits)
        sorted_probs, _ = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_probs, dim=-1)
        nucleus_size = int((cum <= p).sum().item()) + 1
        nucleus_sizes.append(nucleus_size)

        # Sample with top-p
        filtered = top_p_filter(step_logits, p)
        filtered_probs = softmax_from_scratch(filtered)
        next_token = torch.multinomial(filtered_probs, num_samples=1)
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
        tokens_out.append(tokenizer.decode(next_token.item()))
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokens_out, nucleus_sizes

prompt = "In a surprising turn of events, the scientists discovered that"
tokens, sizes = track_nucleus_size(prompt, p=0.9, max_steps=50)

print(f"Generated: {prompt}{''.join(tokens)}\n")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(range(len(sizes)), sizes, color="#6A8EAE", width=0.8)
ax.set_xlabel("Decoding step")
ax.set_ylabel("Nucleus size (tokens)")
ax.set_title(f"Nucleus size per step (top-p = 0.9) — adapts to model confidence")

# Annotate a few tokens
for i in range(0, len(tokens), max(1, len(tokens) // 10)):
    ax.annotate(tokens[i].strip(), (i, sizes[i]), fontsize=6,
                ha="center", va="bottom", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Cumulative probability curves ----------
fig, ax = plt.subplots(figsize=(10, 5))

probs_full = softmax_from_scratch(logits)
sorted_probs, sorted_idx = torch.sort(probs_full, descending=True)
cumulative = torch.cumsum(sorted_probs, dim=-1).cpu().numpy()

# Plot first 200 tokens
n_show = 200
ax.plot(range(n_show), cumulative[:n_show], linewidth=2, color="#4C72B0")

# Draw cutoff lines for different p values
for p_val, color in [(0.5, "#DD5555"), (0.9, "#55AA55"), (0.95, "#E8833A")]:
    cutoff_idx = int((torch.tensor(cumulative) >= p_val).nonzero(as_tuple=True)[0][0].item())
    ax.axhline(y=p_val, color=color, linestyle="--", alpha=0.7)
    ax.axvline(x=cutoff_idx, color=color, linestyle=":", alpha=0.7)
    ax.annotate(f"p={p_val} → {cutoff_idx+1} tokens",
                xy=(cutoff_idx, p_val), fontsize=8, color=color,
                xytext=(cutoff_idx + 10, p_val - 0.05))

ax.set_xlabel("Token rank (sorted by probability)")
ax.set_ylabel("Cumulative probability")
ax.set_title("Cumulative probability curve — where the nucleus cutoff falls")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## Section 6 — Beam Search: Exploring Multiple Paths

Greedy decoding commits to one token at a time and never backtracks. **Beam
search** keeps the **b** most promising *partial sequences* (beams) in parallel:

1. Start with the prompt as the single initial beam.
2. For each beam, compute the next-token probabilities.
3. Expand every beam with its top-b candidates → $b \times b$ candidates.
4. Keep the top-b by *total sequence log-probability*.
5. Repeat until all beams hit EOS or reach `max_length`.

Beam search trades compute for quality: it requires `beam_width` forward passes
per step. It tends to produce fluent, high-probability text — but this is a
double-edged sword. For creative or conversational tasks, "most probable" often
means "most generic".

In [ ]:
def beam_search(prompt: str, beam_width: int = 3, max_new_tokens: int = 50) -> List[Tuple[float, str]]:
    """Beam search implemented from scratch. Returns top-b finished sequences."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

    # Each beam: (log_prob, token_ids_tensor)
    beams = [(0.0, input_ids[0])]
    finished = []

    for step in range(max_new_tokens):
        candidates = []
        for log_prob, seq in beams:
            if seq[-1].item() == tokenizer.eos_token_id:
                finished.append((log_prob, seq))
                continue

            with torch.no_grad():
                out = model(seq.unsqueeze(0))
            step_logits = out.logits[0, -1, :]
            log_probs = torch.log_softmax(step_logits, dim=-1)

            top = torch.topk(log_probs, beam_width)
            for lp, idx in zip(top.values, top.indices):
                new_seq = torch.cat([seq, idx.unsqueeze(0)])
                candidates.append((log_prob + lp.item(), new_seq))

        if not candidates:
            break

        # Keep top beam_width candidates
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

        # Early stop if all beams are finished
        if all(seq[-1].item() == tokenizer.eos_token_id for _, seq in beams):
            finished.extend(beams)
            break

    # Add any unfinished beams
    finished.extend(beams)
    finished.sort(key=lambda x: x[0], reverse=True)

    results = []
    for log_prob, seq in finished[:beam_width]:
        text = tokenizer.decode(seq[input_ids.shape[1]:], skip_special_tokens=True)
        results.append((log_prob, text))
    return results

# Run beam search
prompt = "The key to effective communication is"
print(f"Prompt: \"{prompt}\"\n")
print(f"Beam search (width=3):")
for i, (lp, text) in enumerate(beam_search(prompt, beam_width=3, max_new_tokens=40)):
    print(f"  Beam {i+1} (log-prob: {lp:.2f}): {text.strip()}")

### When beam search helps — and when it hurts

| Scenario | Beam search? | Why |
|---|---|---|
| Machine translation | ✅ Yes | There's typically one "best" translation; beam search finds it. |
| Summarisation | ✅ Often | Summaries should be faithful; high probability ≈ high faithfulness. |
| Structured output (JSON) | ✅ Yes | Syntactic correctness is critical; beam search stays on track. |
| Creative writing | ❌ No | High probability = generic. Sampling produces more interesting text. |
| Open-ended conversation | ❌ No | Beam search responses feel robotic and repetitive. |

**Compute trade-off**: beam search with width $b$ requires roughly $b\times$ the
forward passes of greedy decoding. For interactive applications, this latency
increase is often unacceptable.

---
## Section 7 — Putting It All Together: A Practical Decision Framework

There is no universally correct set of sampling parameters. The right choice
depends on **what you are building**. Here is a starting-point framework:

| Task | Strategy | Temperature | Top-p | Top-k | Notes |
|---|---|---|---|---|---|
| **Code generation** | Sampling | 0.2 | 0.95 | — | Correctness first; slight variation. |
| **Factual QA** | Greedy / low-T | 0.0 – 0.1 | — | — | Deterministic or near-deterministic. |
| **Creative writing** | Sampling | 0.8 – 1.0 | 0.9 | — | Rich vocabulary, surprising choices. |
| **Structured output (JSON)** | Greedy | 0.0 | — | — | Must be syntactically valid. |
| **Brainstorming** | Sampling | 1.2 | 0.95 | — | Push beyond safe defaults. |
| **Summarisation** | Beam search | — | — | — | Beam width 4–6 typical. |
| **Translation** | Beam search | — | — | — | Beam width 4–8 typical. |

> **Key insight**: start with the table above, measure your outputs with an eval
> harness, and iterate. Sampling parameters are hyperparameters — tune them.

In [ ]:
def generate_with_strategy(
    prompt: str,
    strategy: str = "default",
    max_new_tokens: int = 60,
) -> str:
    """Apply a named strategy with sensible defaults.

    Strategies: 'greedy', 'factual', 'code', 'creative', 'brainstorm', 'beam'.
    """
    STRATEGY_PARAMS = {
        "greedy":     {"temperature": None, "top_p": None, "top_k": None, "beam_width": None},
        "factual":    {"temperature": 0.1,  "top_p": None, "top_k": None, "beam_width": None},
        "code":       {"temperature": 0.2,  "top_p": 0.95, "top_k": None, "beam_width": None},
        "creative":   {"temperature": 0.9,  "top_p": 0.9,  "top_k": None, "beam_width": None},
        "brainstorm": {"temperature": 1.2,  "top_p": 0.95, "top_k": None, "beam_width": None},
        "beam":       {"temperature": None, "top_p": None, "top_k": None, "beam_width": 4},
        "default":    {"temperature": 0.7,  "top_p": 0.9,  "top_k": None, "beam_width": None},
    }

    params = STRATEGY_PARAMS.get(strategy, STRATEGY_PARAMS["default"])

    # Beam search path
    if params["beam_width"]:
        results = beam_search(prompt, beam_width=params["beam_width"], max_new_tokens=max_new_tokens)
        return results[0][1] if results else ""

    # Greedy path
    if params["temperature"] is None:
        return greedy_decode(prompt, max_new_tokens=max_new_tokens)

    # Sampling path
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(generated)
        step_logits = out.logits[0, -1, :]

        # Apply temperature
        scaled = step_logits / params["temperature"]

        # Apply top-p if set
        if params["top_p"]:
            scaled = top_p_filter(scaled, params["top_p"])
        # Apply top-k if set
        if params["top_k"]:
            scaled = top_k_filter(scaled, params["top_k"])

        probs = softmax_from_scratch(scaled)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

# Demonstrate all strategies
prompt = "Three tips for writing clean Python code:"
for strategy in ["greedy", "factual", "code", "creative", "brainstorm"]:
    print(f"\n{'='*60}")
    print(f"  Strategy: {strategy}")
    print(f"{'='*60}")
    print(generate_with_strategy(prompt, strategy, max_new_tokens=60))

---
## Section 8 — Repetition Penalties and Stop Sequences

### Repetition penalty

A common problem with autoregressive generation is **degenerate repetition**:
the model gets stuck in a loop. A repetition penalty reduces the logits of
tokens that have already appeared:

$$
z_i' = \begin{cases}
z_i / \alpha & \text{if } z_i > 0 \text{ and token } i \text{ was generated} \\
z_i \times \alpha & \text{if } z_i < 0 \text{ and token } i \text{ was generated} \\
z_i & \text{otherwise}
\end{cases}
$$

where $\alpha > 1$ is the penalty factor. This pushes already-seen tokens away
from being selected, regardless of whether their raw logit is positive or negative.

In [ ]:
def apply_repetition_penalty(
    logits: torch.Tensor,
    generated_ids: torch.Tensor,
    penalty: float = 1.2,
) -> torch.Tensor:
    """Apply repetition penalty to logits for already-generated tokens."""
    penalised = logits.clone()
    unique_ids = generated_ids.unique()

    for token_id in unique_ids:
        if penalised[token_id] > 0:
            penalised[token_id] /= penalty
        else:
            penalised[token_id] *= penalty

    return penalised

def generate_with_rep_penalty(
    prompt: str, temperature: float = 0.7, penalty: float = 1.0,
    max_new_tokens: int = 100,
) -> str:
    """Generate text with optional repetition penalty."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(generated)
        step_logits = out.logits[0, -1, :]

        # Apply repetition penalty on the generated portion only
        if penalty != 1.0:
            gen_portion = generated[0, input_ids.shape[1]:]
            step_logits = apply_repetition_penalty(step_logits, gen_portion, penalty)

        probs = apply_temperature(step_logits, temperature)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

prompt = "The the the the"
print("=== No repetition penalty (penalty=1.0) ===")
print(generate_with_rep_penalty(prompt, temperature=0.7, penalty=1.0, max_new_tokens=80))
print("\n=== With repetition penalty (penalty=1.3) ===")
print(generate_with_rep_penalty(prompt, temperature=0.7, penalty=1.3, max_new_tokens=80))

### Stop sequences

In production, you often want generation to halt when a specific string appears
(e.g., `"\n\n"` for single-paragraph answers, `"```"` for code blocks, or a
custom delimiter like `"<|END|>"`). This is simple to implement: after each
token, check whether the generated text ends with any stop sequence.

In [ ]:
def generate_with_stop(
    prompt: str,
    stop_sequences: List[str],
    temperature: float = 0.7,
    max_new_tokens: int = 200,
) -> str:
    """Generate text and stop when any stop sequence is produced."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(generated)
        step_logits = out.logits[0, -1, :]
        probs = apply_temperature(step_logits, temperature)
        next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)

        text_so_far = tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

        # Check stop sequences
        for stop in stop_sequences:
            if stop in text_so_far:
                # Trim at the stop sequence
                return text_so_far[:text_so_far.index(stop)]

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True)

# Demo: stop at the first newline
result = generate_with_stop(
    prompt="List three fruits:",
    stop_sequences=["\n\n"],
    temperature=0.7,
)
print("Generated (stops at double-newline):")
print(result)

---
## Exercises

### Exercise 1 — Temperature Visualisation

For a prompt of your choice, generate **20 completions** at each of 5 different
temperatures: 0.1, 0.5, 0.7, 1.0, 1.5. For each temperature, compute the
**unique token ratio** (number of unique tokens / total tokens across all
completions). Plot unique token ratio vs. temperature.

*Hint*: use `generate_with_temperature()` and `tokenizer.encode()` to count
tokens.

In [ ]:
# ✏️ Exercise 1 — your code here

# prompt = "The future of artificial intelligence is"
# temperatures = [0.1, 0.5, 0.7, 1.0, 1.5]
# unique_ratios = []
# for T in temperatures:
#     completions = generate_with_temperature(prompt, T, max_new_tokens=30, num_samples=20)
#     all_tokens = []
#     for c in completions:
#         all_tokens.extend(tokenizer.encode(c))
#     ratio = len(set(all_tokens)) / max(len(all_tokens), 1)
#     unique_ratios.append(ratio)
#     print(f"T={T:.1f}  unique ratio = {ratio:.3f}")
#
# plt.figure(figsize=(7, 4))
# plt.plot(temperatures, unique_ratios, "o-", color="#4C72B0")
# plt.xlabel("Temperature")
# plt.ylabel("Unique token ratio")
# plt.title("Output diversity vs. temperature")
# plt.tight_layout()
# plt.show()

### Exercise 2 — Nucleus Size Analysis

For a 100-token generation with top-p = 0.9, record the nucleus size at each
step. Plot nucleus size across steps and annotate the generated tokens.

**Question**: what does a large nucleus at step N tell you about the model's
confidence at that point?

*Hint*: use `track_nucleus_size()` with `max_steps=100`.

In [ ]:
# ✏️ Exercise 2 — your code here

# prompt = "Dear hiring manager, I am writing to express my interest in"
# tokens, sizes = track_nucleus_size(prompt, p=0.9, max_steps=100)
# print(f"Generated: {prompt}{''.join(tokens)}")
# print(f"Average nucleus size: {np.mean(sizes):.1f}")
# print(f"Max nucleus size: {max(sizes)} at step {sizes.index(max(sizes))}")
#
# fig, ax = plt.subplots(figsize=(14, 4))
# ax.bar(range(len(sizes)), sizes, color="#6A8EAE")
# ax.set_xlabel("Step")
# ax.set_ylabel("Nucleus size")
# ax.set_title("Nucleus size per decoding step (p=0.9)")
# plt.tight_layout()
# plt.show()

### Exercise 3 — Decoding Strategy Showdown

Compare **greedy**, **top-k (k=50)**, **top-p (p=0.9)**, and **beam search
(width=3)** on a summarisation-style prompt. Generate the output for each and
evaluate qualitatively: which is most fluent? Most informative? Most generic?

*Hint*: use `greedy_decode()`, `generate_with_strategy()`, and `beam_search()`.

In [ ]:
# ✏️ Exercise 3 — your code here

# prompt = (
#     "Summarise the following in one paragraph: "
#     "Machine learning has transformed industries from healthcare to finance. "
#     "Models can now diagnose diseases, trade stocks, translate languages, and "
#     "generate creative content. However, challenges remain in fairness, "
#     "interpretability, and data privacy.\n\nSummary:"
# )
#
# strategies = {
#     "Greedy": lambda: greedy_decode(prompt, max_new_tokens=80),
#     "Top-k (k=50)": lambda: generate_with_strategy(prompt, "creative", max_new_tokens=80),
#     "Top-p (p=0.9)": lambda: generate_with_strategy(prompt, "default", max_new_tokens=80),
#     "Beam search (w=3)": lambda: beam_search(prompt, beam_width=3, max_new_tokens=80)[0][1],
# }
#
# for name, fn in strategies.items():
#     print(f"\n{'='*60}")
#     print(f"  {name}")
#     print(f"{'='*60}")
#     print(fn())

---
## Key Takeaways

1. **Logits → softmax → probabilities** is the universal first step. Every
   sampling strategy operates on this probability distribution.

2. **Temperature** is the single most impactful parameter. Low T for accuracy,
   high T for creativity. Start at 0.7 and adjust.

3. **Top-k** truncates to a fixed number of candidates. Simple but inflexible.

4. **Top-p (nucleus)** truncates to a fixed probability mass. It adapts to
   model confidence and is the production default in most APIs.

5. **Greedy decoding** (argmax) is deterministic and fast but prone to
   repetition. Use for structured outputs where correctness matters.

6. **Beam search** explores multiple paths in parallel. Best for tasks with a
   single "correct" output (translation, summarisation). Too generic for
   creative tasks.

7. **Repetition penalties** and **stop sequences** are practical necessities
   for production-quality generation.

8. **There is no universally correct setting.** Treat sampling parameters as
   hyperparameters. Measure with evals, then iterate.

## What's Next

- **[04_attention_and_context.ipynb](04_attention_and_context.ipynb)** — How
  attention mechanisms determine what the model "looks at" during generation.
- **Prompt Engineering track** — Now that you understand *how* text is generated,
  learn how to *steer* it through prompting.
- **Systems track** — See how these sampling strategies are implemented
  efficiently in production serving runtimes (vLLM, SGLang).

## References

1. **Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y.** (2020).
   *The Curious Case of Neural Text Degeneration*.
   ICLR 2020. [arXiv:1904.09751](https://arxiv.org/abs/1904.09751) — The
   nucleus (top-p) sampling paper.

2. **Fan, A., Lewis, M., & Dauphin, Y.** (2018). *Hierarchical Neural Story
   Generation*. ACL 2018. [arXiv:1805.04833](https://arxiv.org/abs/1805.04833)
   — Top-k sampling for open-ended generation.

3. **Vijayakumar, A. K., et al.** (2018). *Diverse Beam Search: Decoding
   Diverse Solutions from Neural Sequence Models*.
   [arXiv:1610.02424](https://arxiv.org/abs/1610.02424) — Extensions to
   standard beam search.

4. **Keskar, N. S., McCann, B., Varshney, L. R., Xiong, C., & Socher, R.**
   (2019). *CTRL: A Conditional Transformer Language Model for Controllable
   Generation*. [arXiv:1909.05858](https://arxiv.org/abs/1909.05858) —
   Repetition penalty mechanisms.

5. **Su, Y., et al.** (2022). *A Contrastive Framework for Neural Text
   Generation*. NeurIPS 2022.
   [arXiv:2202.06417](https://arxiv.org/abs/2202.06417) — Contrastive search
   as an alternative decoding strategy.